In [2]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from CRUD_Python_Module import AnimalShelter

###########################
# Data Manipulation / Model
###########################
username = "aacuser"
password = "MYPASSWORD"
shelter = AnimalShelter(username, password)

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(shelter.read({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)
df.drop(columns=['_id'],inplace=True)

## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)

#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

# Loads the Grazioso Salvare Logo image file
image_filename = 'Grazioso Salvare Logo.png' 

# Encodes the image so it can be displayed
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

app.layout = html.Div([

    # Dashboard title
    html.Center(html.B(html.H1('CS-340 Dashboard - Sean Mills'))),
    
    # Creates a logo that can be clicked and takes the user to the SNHU website
    # (would be the Grazioso Salvare website if it was a real website)
    html.Center(
        html.A(
            html.Img(
                # Embedded base64 image source
                src='data:image/png;base64,{}'.format(encoded_image.decode()),
                # Height of the logo
                style={'height': '180px'} 
            ),
            
            # Redirects the user to SNHU's homepage when the logo is clicked
            href='https://www.snhu.edu', 
            
            # Opens the link in a new browser tab
            target='_blank' 
        ) 
    ),
    
    html.Hr(),
    html.Div(
        [
            # Label for the rescue type filter selection
            html.Label("Select Rescue Type:"),
            
            # The radio button options that lets the user filter the dataset
            dcc.RadioItems(
                id='filter-type',
                
                # Defines the rescue category filters that the user selects
                options=[
                    {'label': 'Water Rescue', 'value': 'water'},
                    {'label': 'Mountain or Wilderness Rescue', 'value': 'mountain'},
                    {'label': 'Disaster or Individual Tracking', 'value': 'disaster'},
                    {'label': 'Reset', 'value': 'reset'}
                ],
                
                # Default selection when the dashboard loads
                value='reset',   
                
                # Displays the radio buttons to be vertical, one per line
                labelStyle={'display': 'block'}
            )
        ]   
    ),
    html.Hr(),
    dash_table.DataTable(id='datatable-id',
                         columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
                         data=df.to_dict('records'),
                         row_selectable='single',
                         selected_rows=[0],
                         
                         # Limits the number of animals displayed on the page to 10 and enables pagination
                         page_size=10,

                         # Enables the columns to be sorted
                         sort_action='native',
                         
                         # Enables the columns to be filtered
                         filter_action='native',
                         
                         # Enables horizontal scrolling if the table is wider than the screen
                         style_table={'overflowX': 'auto'},
                        ),
    html.Br(),
    html.Hr(),
    #This sets up the dashboard so that your chart and your geolocation chart are side-by-side
    html.Div(className='row',
         style={'display' : 'flex'},
             children=[
        html.Div(
            id='graph-id',
            className='col s12 m6',

            ),
        html.Div(
            id='map-id',
            className='col s12 m6',
            )
        ])
])

#############################################
# Interaction Between Components / Controller
#############################################
   
@app.callback(Output('datatable-id','data'),
              [Input('filter-type', 'value')])
def update_dashboard(filter_type):
    
    # Applies the filtering logic based on the selected rescue type
    if filter_type == 'water':
        query = {
            "$and": [
                {
                    "$or": [
                        # $regex is used for pattern matching on the breed names
                        # "i" option makes the search case-insensitive
                        {"breed": {"$regex": "Labrador Retriever", "$options": "i"}},
                        {"breed": {"$regex": "Chesapeake Bay Retriever", "$options": "i"}},
                        {"breed": {"$regex": "Newfoundland", "$options": "i"}}
                    ]
                },
                {"sex_upon_outcome": "Intact Female"},
                
                # Filter for dogs with ages between 26 and 156 weeks
                {"age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}}
            ]
        }

    elif filter_type == 'mountain':
        query = {
            "$and": [
                {
                    "$or": [
                        {"breed": {"$regex": "German Shepherd", "$options": "i"}},
                        {"breed": {"$regex": "Alaskan Malamute", "$options": "i"}},
                        {"breed": {"$regex": "Old English Sheepdog", "$options": "i"}},
                        {"breed": {"$regex": "Siberian Husky", "$options": "i"}},
                        {"breed": {"$regex": "Rottweiler", "$options": "i"}}
                    ]
                },
                {"sex_upon_outcome": "Intact Male"},
                
                # Ages between 26 and 156 weeks
                {"age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}}
            ]
        }
    
    elif filter_type == 'disaster':
        query = {
            "$and": [
                {
                    "$or": [
                        {"breed": {"$regex": "Doberman Pinscher", "$options": "i"}},
                        {"breed": {"$regex": "German Shepherd", "$options": "i"}},
                        {"breed": {"$regex": "Golden Retriever", "$options": "i"}},
                        {"breed": {"$regex": "Bloodhound", "$options": "i"}},
                        {"breed": {"$regex": "Rottweiler", "$options": "i"}}
                    ]
                },
                {"sex_upon_outcome": "Intact Male"},
                
                # Ages between 20 and 300 weeks
                {"age_upon_outcome_in_weeks": {"$gte": 20, "$lte": 300}}
            ]
        }

    else:
        # Reset
        query = {}
        
    filtered_df = pd.DataFrame.from_records(shelter.read(query))
    
    if '_id' in filtered_df.columns:
        filtered_df.drop(columns=['_id'], inplace=True)
       
    # Filters out duplicate records
    if 'animal_id' in filtered_df.columns:
        filtered_df = filtered_df.drop_duplicates(subset=['animal_id'])
    
    return filtered_df.to_dict('records')

# Display the breeds of animal based on quantity represented in
# the data table
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")])
def update_graphs(viewData):
    
    if viewData is None:
        return
    
    dff = pd.DataFrame(viewData)
    
    if dff.empty:
        return html.H4("No data is available for the selected filter.")
    
    # Count the number of breeds
    breed_count = dff['breed'].value_counts().reset_index()
    breed_count.columns = ['breed', 'count']
    
    # Only show the top 10 breeds
    top_breeds = breed_count.head(10)
    
    fig = px.pie(
        top_breeds,
        names='breed',
        values='count',
        title='Top 10 Breed Distribution')
    
    # Centers the chart title over the chart
    fig.update_layout(title_x=0.27)
    
    return [ 
        dcc.Graph(figure=fig) 
    ]
       
#This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    
    if selected_columns is None:
        return[]
    
    return [{
        'if': { 'column_id': i },
        'background_color': '#D2F3FF'
    } for i in selected_columns]

# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")])

def update_map(viewData, index):  
    if viewData is None:
        return
    elif index is None:
        return
    
    dff = pd.DataFrame.from_dict(viewData)
    
    if dff.empty:
        return
    
    # If no row is selected default to the first row
    if not index:
        row = 0
    else: 
        row = index[0]
        
    # Column 13 and 14 define the grid-coordinates for the map
    # Column 4 defines the breed for the animal
    # Column 9 defines the name of the animal
    
    # If the animal has no name, display "N/A" on the map so the user knows
    animal_name = dff.iloc[row, 9]
    if pd.isna(animal_name) or str(animal_name).strip() == "":
        animal_name = "N/A"
      
    # Gets the latitude and longitude for the selected animal
    lat = dff.iloc[row, 13]
    lon = dff.iloc[row, 14]
    
    # Map dynamically centers on the selected animal's location so that it updates
    # whenever a different row is selected
    return [
        dl.Map(style={'width': '1000px', 'height': '500px'}, center=[lat, lon], zoom=10, children=[
            dl.TileLayer(id="base-layer-id"),
            # Marker with tool tip and popup
            dl.Marker(position=[lat, lon], children=[
                dl.Tooltip(dff.iloc[row,4]),
                dl.Popup([
                    html.H1("Animal Name"),
                    html.P(animal_name)
                ])
            ])
        ])
    ]

# Run app and display result in jupyterlab mode, note, if you have previously run a prior app, the default port of 8050 may not be available, if so, try setting an alternate port.
app.run_server(mode='external', port=8051) #Alternate port 8051

Dash app running on https://ferrarichild-ricardohorizon-3000.codio.io/proxy/8051/
